In [19]:
import pandas as pd
import numpy as np
import os 
import glob
from pathlib import Path
import calendar
import warnings

warnings.filterwarnings('ignore')  # Silence benign pandas warnings

print("✅ All libraries imported successfully.")
print(f"   pandas  version: {pd.__version__}")
print(f"   numpy   version: {np.__version__}")



✅ All libraries imported successfully.
   pandas  version: 2.3.3
   numpy   version: 2.2.5


Set Paths and Load stations_info.csv

In [20]:

DATA_DIR          = Path("../data")
OUTPUT_DIR        = Path("../outputs")
FINAL_DATASET_DIR = OUTPUT_DIR / "final_dataset"   # Per-city CSVs go here

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DATASET_DIR.mkdir(parents=True, exist_ok=True)

print(f" Data folder       : {DATA_DIR.resolve()}")
print(f" Output folder     : {OUTPUT_DIR.resolve()}")
print(f" Per-city folder   : {FINAL_DATASET_DIR.resolve()}")

stations_info_path = DATA_DIR / "stations_info.csv"

if not stations_info_path.exists():
    raise FileNotFoundError(f"Cannot find stations_info.csv at {stations_info_path}")

stations_info = pd.read_csv(stations_info_path)

print(f"\n✅ Loaded stations_info.csv")
print(f"   Rows (stations): {len(stations_info)}")
print(f"   Columns: {list(stations_info.columns)}")
print(f"\nFirst 3 rows:")
stations_info.head(3)


 Data folder       : C:\Users\GARV VERMA\Desktop\Storage\codes\Projects\Pollution_Project\data
 Output folder     : C:\Users\GARV VERMA\Desktop\Storage\codes\Projects\Pollution_Project\outputs
 Per-city folder   : C:\Users\GARV VERMA\Desktop\Storage\codes\Projects\Pollution_Project\outputs\final_dataset

✅ Loaded stations_info.csv
   Rows (stations): 453
   Columns: ['file_name', 'state', 'city', 'agency', 'station_location', 'start_month', 'start_month_num', 'start_year']

First 3 rows:


,file_name,state,city,agency,station_location,start_month,start_month_num,start_year
0,AP001,Andhra Pradesh,Tirupati,APPCB,"Tirumala, Tirupati",July,7,2016
1,AP002,Andhra Pradesh,Vijayawada,APPCB,"PWD Grounds, Vijayawada",May,5,2017
2,AP003,Andhra Pradesh,Visakhapatnam,APPCB,"GVM Corporation, Visakhapatnam",July,7,2017


City to stations Dictionary

In [21]:
city_to_stations = {}  
station_to_meta  = {}    

for _, row in stations_info.iterrows():
    station_id = str(row['file_name']).replace('.csv', '').strip()
    city       = str(row['city']).strip()

    if city not in city_to_stations:
        city_to_stations[city] = []
    city_to_stations[city].append(station_id)

    station_to_meta[station_id] = {
        'state'            : row['state'],
        'city'             : row['city'],
        'agency'           : row['agency'],
        'station_location' : row['station_location'],
        'start_year'       : row['start_year'],
        'start_month_num'  : row['start_month_num'],
    }

# ── Summary ──
n_cities   = len(city_to_stations)
n_stations = len(station_to_meta)
multi      = {c: s for c, s in city_to_stations.items() if len(s) > 1}

print(f"✅ Dictionary built")
print(f"   Unique cities   : {n_cities}")
print(f"   Total stations  : {n_stations}")
print(f"   Cities with 2+ stations: {len(multi)}")

if multi:
    print("\n   Top multi-station cities:")
    for city, stns in sorted(multi.items(), key=lambda x: -len(x[1]))[:15]:
        print(f"     {city:35s}: {stns}")


✅ Dictionary built
   Unique cities   : 241
   Total stations  : 453
   Cities with 2+ stations: 55

   Top multi-station cities:
     Delhi                              : ['DL001', 'DL002', 'DL003', 'DL004', 'DL005', 'DL006', 'DL007', 'DL008', 'DL009', 'DL010', 'DL011', 'DL012', 'DL013', 'DL014', 'DL015', 'DL016', 'DL017', 'DL018', 'DL019', 'DL020', 'DL021', 'DL022', 'DL023', 'DL024', 'DL025', 'DL026', 'DL027', 'DL028', 'DL029', 'DL030', 'DL031', 'DL032', 'DL033', 'DL034', 'DL035', 'DL036', 'DL037', 'DL038', 'DL039', 'DL040']
     Mumbai                             : ['MH002', 'MH013', 'MH014', 'MH015', 'MH016', 'MH017', 'MH018', 'MH019', 'MH020', 'MH021', 'MH022', 'MH023', 'MH025', 'MH026', 'MH029', 'MH033', 'MH034', 'MH035', 'MH036', 'MH037', 'MH039']
     Hyderabad                          : ['TG001', 'TG002', 'TG003', 'TG004', 'TG005', 'TG006', 'TG007', 'TG008', 'TG009', 'TG010', 'TG011', 'TG012', 'TG013', 'TG014']
     Bengaluru                          : ['KA001', 'KA002', 'KA00

In [22]:
# ── DEDUPLICATE_MAP ──────────────────────────────────────────────────────────
# Format: 'messy_original_name' → 'clean_canonical_name'
# Every messy variant found during Phase 0's column audit is listed here.

DEDUPLICATE_MAP = {

    # ── NOx variants ──────────────────────────────────────────────────────────
    'NOx (ug/m3)'      : 'NOx (ppb)',
    'NOx (ppm)'        : 'NOx (ppb)',

    # ── CO variants ───────────────────────────────────────────────────────────
    'CO (ug/m3)'       : 'CO (mg/m3)',
    'CO (mg/Nm3)'      : 'CO (mg/m3)',
    'CO (ng/m3)'       : 'CO (mg/m3)',

    # ── NO variants ───────────────────────────────────────────────────────────
    'NO (mg/m3)'       : 'NO (ug/m3)',
    'NO (ppb)'         : 'NO (ug/m3)',
    'NO (ppm)'         : 'NO (ug/m3)',
    'NO ()'            : 'NO (ug/m3)',

    # ── Ozone variants ────────────────────────────────────────────────────────
    'Ozone ()'         : 'Ozone (ug/m3)',
    'Ozone (ppb)'      : 'Ozone (ug/m3)',

    # ── NH3 variants ──────────────────────────────────────────────────────────
    'NH3 ()'           : 'NH3 (ug/m3)',
    'NH3 (ppb)'        : 'NH3 (ug/m3)',

    # ── SO2 variants ──────────────────────────────────────────────────────────
    'SO2 ()'           : 'SO2 (ug/m3)',

    # ── Temperature variants (all → AT (degree C)) ────────────────────────────
    # Some files mislabelled temperature column with wrong units — still temperature
    'AT ()'            : 'AT (degree C)',
    'AT (degree)'      : 'AT (degree C)',
    'AT (ug/m3)'       : 'AT (degree C)',   # Labelling error in raw data
    'Temp (degree C)'  : 'AT (degree C)',
    'Temp ()'          : 'AT (degree C)',
    'Temp (ug/m3)'     : 'AT (degree C)',   # Labelling error in raw data

    # ── Relative Humidity variants ────────────────────────────────────────────
    'RH ()'            : 'RH (%)',
    'RH (W/mt2)'       : 'RH (%)',          # Labelling error
    'RH (degree)'      : 'RH (%)',          # Labelling error

    # ── Wind Speed variants ───────────────────────────────────────────────────
    'WS ()'            : 'WS (m/s)',
    'WS (ug/m3)'       : 'WS (m/s)',        # Labelling error
    'VWS (m/s)'        : 'WS (m/s)',        # VWS = same as WS here

    # ── Wind Direction variants ───────────────────────────────────────────────
    'WD (deg)'         : 'WD (degree)',
    'WD (degree C)'    : 'WD (degree)',     # Labelling error
    'WD ()'            : 'WD (degree)',

    # ── Solar Radiation variants ──────────────────────────────────────────────
    'SR ()'            : 'SR (W/mt2)',
    'SR (ug/m3)'       : 'SR (W/mt2)',      # Labelling error

    # ── Rainfall variants (including pandas auto-deduplicated duplicate columns)
    # When a CSV has the same column name twice, pandas adds .1 .2 etc automatically
    'RF ()'            : 'RF (mm)',
    'RF (m/s)'         : 'RF (mm)',         # Labelling error
    'RF (mm).1'        : 'RF (mm)',
    'RF (mm).2'        : 'RF (mm)',
    'RF (mm).3'        : 'RF (mm)',
    'RF (mm).4'        : 'RF (mm)',
    'RF (mm).5'        : 'RF (mm)',
    'RF (mm).6'        : 'RF (mm)',
    'RF (mm).7'        : 'RF (mm)',

    # ── Barometric Pressure variants ──────────────────────────────────────────
    'BP ()'            : 'BP (mmHg)',
    'BP (mg/m3)'       : 'BP (mmHg)',       # Labelling error
    'BP (W/mt2)'       : 'BP (mmHg)',       # Labelling error
    'BP (mmHg).1'      : 'BP (mmHg)',
    'BP (mmHg).2'      : 'BP (mmHg)',
    'BP (mmHg).3'      : 'BP (mmHg)',
    'BP (mmHg).4'      : 'BP (mmHg)',
    'BP (mmHg).5'      : 'BP (mmHg)',
    'BP (mmHg).6'      : 'BP (mmHg)',
    'BP (mmHg).7'      : 'BP (mmHg)',

    # ── VOC variants ──────────────────────────────────────────────────────────
    'Benzene ()'       : 'Benzene (ug/m3)',
    'Benzene (mg/m3)'  : 'Benzene (ug/m3)',
    'Toluene ()'       : 'Toluene (ug/m3)',
    'Eth-Benzene ()'   : 'Eth-Benzene (ug/m3)',
    'MP-Xylene ()'     : 'MP-Xylene (ug/m3)',
    'Xylene ()'        : 'Xylene (ug/m3)',

    # ── Other rare variants ───────────────────────────────────────────────────
    'CH4 ()'           : 'CH4 (ug/m3)',
    'NMHC ()'          : 'NMHC (ug/m3)',
    'THC ()'           : 'THC (ug/m3)',
}

print(f"✅ DEDUPLICATE_MAP defined: {len(DEDUPLICATE_MAP)} renaming rules")


✅ DEDUPLICATE_MAP defined: 59 renaming rules


Define Unit Conversion Map and Function

After renaming, all columns named `NOx (ppb)` should contain values in ppb.
But some of those rows came from files where the original label was `NOx (ppm)` —
those numbers are 1000× too small.

**We must multiply/divide the actual numbers to match the canonical unit.**

This uses the original column name (before renaming) to know which conversion to apply.
Conversion factors are sourced from standard atmospheric chemistry at 25°C, 1 atm.

| Original column     | Target        | Conversion                          |
|---------------------|---------------|-------------------------------------|
| NOx (ppm)           | NOx (ppb)     | × 1000                              |
| NOx (ug/m3)         | NOx (ppb)     | × 0.531  (NO₂ MW=46 at 25°C)        |
| CO (ug/m3)          | CO (mg/m3)    | ÷ 1000                              |
| CO (ng/m3)          | CO (mg/m3)    | ÷ 1,000,000                         |
| CO (mg/Nm3)         | CO (mg/m3)    | no change (≈ equivalent)            |
| NO (ppb)            | NO (ug/m3)    | × 1.23   (NO MW=30 at 25°C)         |
| NO (ppm)            | NO (ug/m3)    | × 1230                              |
| NO (mg/m3)          | NO (ug/m3)    | × 1000                              |
| NH3 (ppb)           | NH3 (ug/m3)   | × 0.71   (NH₃ MW=17 at 25°C)        |
| Ozone (ppb)         | Ozone (ug/m3) | × 1.96   (O₃ MW=48 at 25°C)         |
| Benzene (mg/m3)     | Benzene(ug/m3)| × 1000                              |

These conversions affect fewer than 3% of files each — small but important for accuracy.


In [23]:
# ── UNIT_CONVERSION_MAP ──────────────────────────────────────────────────────
# Format: 'original_column_name' → ('operation', factor)
# Applied BEFORE renaming — we use the original name to know what conversion needed

UNIT_CONVERSION_MAP = {

    # NOx: some files used ppm or ug/m3, but canonical is ppb
    'NOx (ppm)'        : ('multiply',  1000.0),    # ppm × 1000 = ppb
    'NOx (ug/m3)'      : ('multiply',  0.531),     # NO2 MW=46; 1 ug/m3 = 0.531 ppb at 25°C

    # CO: canonical is mg/m3
    'CO (ug/m3)'       : ('divide',    1000.0),    # ug ÷ 1000 = mg
    'CO (ng/m3)'       : ('divide',    1_000_000), # ng ÷ 1,000,000 = mg
    
    # CO (mg/Nm3) ≈ CO (mg/m3) at ambient conditions → no conversion needed

    # NO: canonical is ug/m3
    'NO (ppb)'         : ('multiply',  1.23),      # NO MW=30; 1 ppb = 1.23 ug/m3 at 25°C
    'NO (ppm)'         : ('multiply',  1230.0),    # ppm × 1000 × 1.23
    'NO (mg/m3)'       : ('multiply',  1000.0),    # mg × 1000 = ug

    # NH3: canonical is ug/m3
    'NH3 (ppb)'        : ('multiply',  0.71),      # NH3 MW=17; 1 ppb = 0.71 ug/m3 at 25°C

    # Ozone: canonical is ug/m3
    'Ozone (ppb)'      : ('multiply',  1.96),      # O3 MW=48; 1 ppb = 1.96 ug/m3 at 25°C

    # Benzene: canonical is ug/m3
    'Benzene (mg/m3)'  : ('multiply',  1000.0),    # mg × 1000 = ug
}

print(f"✅ UNIT_CONVERSION_MAP defined: {len(UNIT_CONVERSION_MAP)} conversion rules")
print()
print("Conversion summary:")
for col, (op, factor) in UNIT_CONVERSION_MAP.items():
    print(f"  {col:25s}  →  {op} by {factor}")


# ── apply_unit_conversions() ─────────────────────────────────────────────────
def apply_unit_conversions(df, original_columns):

    conversions_applied = []

    for orig_col in original_columns:
        if orig_col in UNIT_CONVERSION_MAP and orig_col in df.columns:
            operation, factor = UNIT_CONVERSION_MAP[orig_col]

            if operation == 'multiply':
                # WHY: values in this column are in wrong unit, multiply to convert
                df[orig_col] = pd.to_numeric(df[orig_col], errors='coerce') * factor
            elif operation == 'divide':
                # WHY: values are too large by this factor, divide to convert
                df[orig_col] = pd.to_numeric(df[orig_col], errors='coerce') / factor

            conversions_applied.append(f"{orig_col} ({operation} × {factor})")

    return df, conversions_applied

print("\n✅ apply_unit_conversions() function defined")


✅ UNIT_CONVERSION_MAP defined: 10 conversion rules

Conversion summary:
  NOx (ppm)                  →  multiply by 1000.0
  NOx (ug/m3)                →  multiply by 0.531
  CO (ug/m3)                 →  divide by 1000.0
  CO (ng/m3)                 →  divide by 1000000
  NO (ppb)                   →  multiply by 1.23
  NO (ppm)                   →  multiply by 1230.0
  NO (mg/m3)                 →  multiply by 1000.0
  NH3 (ppb)                  →  multiply by 0.71
  Ozone (ppb)                →  multiply by 1.96
  Benzene (mg/m3)            →  multiply by 1000.0

✅ apply_unit_conversions() function defined


Deciding which columns to keep and which to drop

In [24]:
# ── Columns to DROP completely ────────────────────────────────────────────────
# These are station hardware readings — not environmental measurements.
# They do not represent pollution or weather — they are electrical/mechanical
# diagnostics of the monitoring equipment itself.
COLUMNS_TO_DROP = ['Variance(n)', 'Power(W)', 'MH(m)', 'VWS(degree)']

METADATA_COLS = ['From Date', 'To Date']


CORE_POLLUTANT_COLS = [
    'PM2.5 (ug/m3)', 'PM10 (ug/m3)',
    'NO (ug/m3)', 'NO2 (ug/m3)', 'NOx (ppb)',
    'SO2 (ug/m3)', 'CO (mg/m3)', 'NH3 (ug/m3)', 'Ozone (ug/m3)',
]

VOC_COLS = [
    'Benzene (ug/m3)', 'Toluene (ug/m3)', 'Eth-Benzene (ug/m3)',
    'MP-Xylene (ug/m3)', 'O Xylene (ug/m3)', 'Xylene (ug/m3)',
]

WEATHER_COLS = [
    'WS (m/s)', 'WD (degree)', 'RH (%)', 'AT (degree C)',
    'SR (W/mt2)', 'RF (mm)', 'BP (mmHg)',
]

MINOR_COLS = [
    'CH4 (ug/m3)', 'NMHC (ug/m3)', 'THC (ug/m3)',
    'CO2 (mg/m3)', 'HCHO (ug/m3)', 'Hg (ug/m3)', 'SPM (ug/m3)',
]

ALL_MEASUREMENT_COLS = CORE_POLLUTANT_COLS + VOC_COLS + WEATHER_COLS + MINOR_COLS

print("✅ Column lists defined")
print(f"   Core pollutants : {len(CORE_POLLUTANT_COLS)}")
print(f"   VOCs            : {len(VOC_COLS)}")
print(f"   Weather         : {len(WEATHER_COLS)}")
print(f"   Minor/rare      : {len(MINOR_COLS)}")
print(f"   Total measurement columns expected: {len(ALL_MEASUREMENT_COLS)}")

✅ Column lists defined
   Core pollutants : 9
   VOCs            : 6
   Weather         : 7
   Minor/rare      : 7
   Total measurement columns expected: 29


 Define Monthly Aggregation Function

| % of hours with real data | monthly mean | flag          |
|---------------------------|--------------|---------------|
| ≥ 50%                     | computed     | `good`        |
| 25 – 49%                  | computed     | `sparse`      |
| < 25%                     | NaN          | `insufficient`|


In [25]:
# ── Robust date parser (handles all format variants seen in CPCB data) ────────
# WHY THIS FUNCTION EXISTS:
# Different station CSVs use different date formats. Some use dashes, some use
# slashes, some include seconds, some don't. A single pd.to_datetime() call
# with dayfirst=True only handles ONE format — all others return NaT and get
# silently dropped, which is what caused the massive row losses in the loop.
# This function tries every known format until one works for a given column.

KNOWN_DATE_FORMATS = [
    '%d-%m-%Y %H:%M',      # "01-01-2019 00:00"   ← most common CPCB format
    '%d/%m/%Y %H:%M',      # "01/01/2019 00:00"   ← slash variant
    '%d-%m-%Y %H:%M:%S',   # "01-01-2019 00:00:00" ← with seconds
    '%d/%m/%Y %H:%M:%S',   # "01/01/2019 00:00:00"
    '%Y-%m-%d %H:%M:%S',   # "2019-01-01 00:00:00" ← ISO format
    '%Y-%m-%d %H:%M',      # "2019-01-01 00:00"
    '%m/%d/%Y %H:%M',      # "01/01/2019 00:00" US style 
    '%d %b %Y %H:%M',      # "01 Jan 2019 00:00"
]

def parse_date_column(series):
    sample = series.dropna().head(200)
    
    if len(sample) == 0:
        return pd.to_datetime(series, errors='coerce')
    
    best_format    = None
    best_success   = 0
    
    for fmt in KNOWN_DATE_FORMATS:
        try:
            parsed_sample = pd.to_datetime(sample, format=fmt, errors='coerce')
            
            success_count = parsed_sample.notna().sum()
            success_rate  = success_count / len(sample)
            
            if success_rate > best_success:
                best_success = success_rate
                best_format  = fmt
                
            
            if success_rate >= 0.95:
                break
        except Exception:
            continue
    
    if best_format is not None and best_success >= 0.50:
       
        parsed = pd.to_datetime(series, format=best_format, errors='coerce')
    else:

        parsed = pd.to_datetime(series, infer_datetime_format=True, errors='coerce')
    
    return parsed

def hours_in_month(year, month):
    """Returns the total number of hours in a given year+month."""
    days = calendar.monthrange(int(year), int(month))[1]
    return days * 24

def aggregate_hourly_to_monthly(df_city, city_name, n_stations):

    df_city = df_city.copy()
    
    # ── Robust date parsing — tries all known formats ────
    df_city['From Date'] = parse_date_column(df_city['From Date'])
 
    n_bad_dates = df_city['From Date'].isna().sum()
    if n_bad_dates > 0:
        pct = n_bad_dates / len(df_city) * 100
        if pct > 1.0:  
            print(f"   ⚠️  {n_bad_dates:,} rows ({pct:.1f}%) had unparseable dates — dropped")
        df_city = df_city.dropna(subset=['From Date'])
    
    if len(df_city) == 0:
        return pd.DataFrame()
    
    # Extract year and month
    df_city['year']  = df_city['From Date'].dt.year
    df_city['month'] = df_city['From Date'].dt.month
    
    # ── Identify measurement columns present in this city's data ─────
    measurement_cols_present = [c for c in ALL_MEASUREMENT_COLS if c in df_city.columns]
    
    # ── Aggregate each (year, month) group ───
    monthly_rows = []
    grouped = df_city.groupby(['year', 'month'])
    
    for (year, month), group in grouped:
        
        possible = hours_in_month(int(year), int(month)) * n_stations
        row = {'year': int(year), 'month': int(month)}
        
        for col in measurement_cols_present:
            actual_count = group[col].notna().sum()
            coverage     = min(actual_count / possible, 1.0) if possible > 0 else 0.0
            
            if coverage >= 0.50:
                row[col + '_mean'] = group[col].mean()
                row[col + '_flag'] = 'good'
            elif coverage >= 0.25:
                row[col + '_mean'] = group[col].mean()
                row[col + '_flag'] = 'sparse'
            else:
                row[col + '_mean'] = np.nan
                row[col + '_flag'] = 'insufficient'
            
            row[col + '_coverage_pct'] = round(coverage * 100, 1)
        
    
        for col in ALL_MEASUREMENT_COLS:
            if col not in measurement_cols_present:
                row[col + '_mean']         = np.nan
                row[col + '_flag']         = 'no_data'
                row[col + '_coverage_pct'] = 0.0
        
        monthly_rows.append(row)
    
    return pd.DataFrame(monthly_rows)


print("✅ parse_date_column() and aggregate_hourly_to_monthly() defined")
print(f"   Will try {len(KNOWN_DATE_FORMATS)} date formats before falling back to inference")
print("   Warnings only printed if >1% of rows in a file are unparseable")

✅ parse_date_column() and aggregate_hourly_to_monthly() defined
   Will try 8 date formats before falling back to inference
   Warnings only printed if >1% of rows in a file are unparseable


Define Tier Assignment Function

Each city gets a data quality tier based on how many years of actual data it has.

| Tier   | Years of data | What it allows                         |
|--------|---------------|----------------------------------------|
| tier_1 | ≥ 8 years     | Full trend analysis + ML training      |
| tier_2 | 5–7 years     | Trend analysis + limited ML            |
| tier_3 | 3–4 years     | Snapshot / recent trend only           |
| tier_4 | < 3 years     | Flagged — excluded from ML and trends  |

**Why not delete tier_4 cities?** We keep all data. Low-tier cities are still
useful for AQI maps and national health dashboards. We just don't run ML on them.

The actual year span is calculated from the real data (first usable month → last usable month),
not from the nominal start year in stations_info.csv (which can be misleading, as UP001 showed).


In [26]:
def assign_tier(monthly_df):
    pm25_col = 'PM2.5 (ug/m3)_flag'

    if pm25_col not in monthly_df.columns:
     
        usable_years = sorted(monthly_df['year'].unique())
    else:

        usable_mask = monthly_df[pm25_col].isin(['good', 'sparse'])
        usable_years = sorted(monthly_df.loc[usable_mask, 'year'].unique())

    if len(usable_years) == 0:
  
        any_good = monthly_df[[c for c in monthly_df.columns if c.endswith('_flag')]]
        any_good = any_good.isin(['good', 'sparse'])
        usable_years_any = sorted(monthly_df.loc[any_good.any(axis=1), 'year'].unique())
        if len(usable_years_any) == 0:
            return 'tier_4', 0, None, None
        usable_years = usable_years_any

    first_year   = int(usable_years[0])
    last_year    = int(usable_years[-1])
    actual_years = last_year - first_year + 1

    if actual_years >= 8:
        tier = 'tier_1'
    elif actual_years >= 5:
        tier = 'tier_2'
    elif actual_years >= 3:
        tier = 'tier_3'
    else:
        tier = 'tier_4'

    return tier, actual_years, first_year, last_year


print("✅ assign_tier() function defined")
print("   tier_1: ≥8yr | tier_2: 5-7yr | tier_3: 3-4yr | tier_4: <3yr")


✅ assign_tier() function defined
   tier_1: ≥8yr | tier_2: 5-7yr | tier_3: 3-4yr | tier_4: <3yr


State Name Standardisation

In [27]:
# ── STATE NAME STANDARDISATION MAP ───────────────────────────────────────────
STATE_NAME_MAP = {
    'Jammu and Kashmir' : 'Jammu & Kashmir and Ladakh',  
    'Chandigarh'        : 'Other Union Territories',     
    'Puducherry'        : 'Other Union Territories',   
}

STATES_ALREADY_MATCHING = [
    'Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chhattisgarh',
    'Delhi', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jharkhand', 'Karnataka',
    'Kerala', 'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram',
    'Nagaland', 'Odisha', 'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu',
    'Telangana', 'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal',
]

print("✅ STATE_NAME_MAP defined")
print(f"   States requiring renaming    : {len(STATE_NAME_MAP)}")
print(f"   States already matching      : {len(STATES_ALREADY_MATCHING)}")
print()
print("Renames to apply:")
for old, new in STATE_NAME_MAP.items():
    print(f"  '{old}'  →  '{new}'")


✅ STATE_NAME_MAP defined
   States requiring renaming    : 3
   States already matching      : 28

Renames to apply:
  'Jammu and Kashmir'  →  'Jammu & Kashmir and Ladakh'
  'Chandigarh'  →  'Other Union Territories'
  'Puducherry'  →  'Other Union Territories'


In [28]:
# ── Helper: collapse any remaining duplicate column names ─────────────────────
# WHY THIS FUNCTION EXISTS:
# After DEDUPLICATE_MAP renames columns, two originally-different columns can
# end up with the same name. Example: a file has both 'RF (mm)' and 'RF (m/s)'.
# After renaming, both become 'RF (mm)'. Now the DataFrame has two columns called
# 'RF (mm)'. pd.concat() cannot handle this — it crashes with InvalidIndexError.
# This function detects any such duplicates and averages them into a single column.

def collapse_duplicate_columns(df):
    """
    WHY: After renaming, duplicate column names must be merged before concat.
    For measurement columns, averaging is the right approach (both columns
    measure the same thing — we just had two copies of it).
    For metadata/string columns (station_id, city etc.), we keep the first.
    """
    if df.columns.is_unique:
        return df   # No duplicates — nothing to do, return as-is

    # Find which column names appear more than once
    col_counts = {}
    for col in df.columns:
        col_counts[col] = col_counts.get(col, 0) + 1
    duplicate_names = [col for col, count in col_counts.items() if count > 1]

    for col in duplicate_names:
        # Extract all columns that share this name (could be 2, 3, or more)
        all_copies = df.loc[:, df.columns == col]

        # Try to treat them as numeric and average them
        # errors='coerce' turns any non-numeric values into NaN silently
        numeric_copies = all_copies.apply(pd.to_numeric, errors='coerce')

        # Remove all copies of this column from the DataFrame
        df = df.loc[:, df.columns != col].copy()

        # Add back a single averaged column
        # skipna=True means: if one copy is NaN but another has a value, use the value
        df[col] = numeric_copies.mean(axis=1, skipna=True)

    return df


def load_and_clean_station(station_id):
    """
    Loads one station CSV, cleans it, and returns a tidy DataFrame.
    Steps:
      1. Load CSV
      2. Apply unit conversions (must happen BEFORE renaming)
      3. Apply DEDUPLICATE_MAP renaming
      4. Collapse any duplicate column names that renaming created  ← NEW STEP
      5. Drop hardware columns
      6. Add station metadata columns
    """
    csv_path = DATA_DIR / f"{station_id}.csv"

    if not csv_path.exists():
        print(f"   ⚠️  File not found: {csv_path}")
        return None

    # Load CSV — low_memory=False reads whole file to detect dtypes correctly
    df = pd.read_csv(csv_path, low_memory=False)

    # Save original column names BEFORE any changes
    # WHY: unit conversion keys are original names — we need them unchanged
    original_columns = list(df.columns)

    # Step 1: Apply unit value conversions (using original names as keys)
    df, conversions_done = apply_unit_conversions(df, original_columns)

    # Step 2: Rename messy column names → canonical names
    df.rename(columns=DEDUPLICATE_MAP, inplace=True)

    # Step 3: Collapse duplicate column names that renaming may have created
    # WHY: This is the root cause of the InvalidIndexError in pd.concat.
    # Must happen HERE, inside this function, before the DataFrame leaves.
    df = collapse_duplicate_columns(df)

    # Step 4: Drop hardware/irrelevant columns
    cols_to_drop_now = [c for c in COLUMNS_TO_DROP if c in df.columns]
    if cols_to_drop_now:
        df.drop(columns=cols_to_drop_now, inplace=True)

    # Step 5: Add station metadata columns
    meta = station_to_meta.get(station_id, {})
    df['station_id']       = station_id
    df['state']            = meta.get('state', 'Unknown')
    df['city']             = meta.get('city', 'Unknown')
    df['agency']           = meta.get('agency', 'Unknown')
    df['station_location'] = meta.get('station_location', 'Unknown')

    return df


print("✅ collapse_duplicate_columns() and load_and_clean_station() defined")
print("   Duplicate column merging will now happen inside each station load.")
print("   This prevents the InvalidIndexError in pd.concat.")

✅ collapse_duplicate_columns() and load_and_clean_station() defined
   Duplicate column merging will now happen inside each station load.
   This prevents the InvalidIndexError in pd.concat.


Main Proccessing Loop for all Cities

In [29]:
# ── Main processing loop ──────────────────────────────────────────────────────
# We collect monthly DataFrames for all cities here, then combine at the end
all_city_monthly_dfs = []   # List of per-city monthly DataFrames

# Track statistics for the summary report
cities_processed   = 0
cities_skipped     = 0
stations_not_found = []

city_list = sorted(city_to_stations.keys())   # Sort alphabetically for reproducibility
n_total_cities = len(city_list)

print(f"Starting main loop: {n_total_cities} cities to process")
print("=" * 60)

for city_idx, city_name in enumerate(city_list):

    station_ids = city_to_stations[city_name]
    n_stations  = len(station_ids)

    # ── Load all stations for this city ──────────────────────────────────────
    city_station_dfs = []

    for station_id in station_ids:
        df_station = load_and_clean_station(station_id)
        if df_station is not None:
            city_station_dfs.append(df_station)
        else:
            stations_not_found.append(station_id)

    # If no station files were found for this city, skip it
    if len(city_station_dfs) == 0:
        print(f"  ⚠️  SKIP {city_name}: no station files found")
        cities_skipped += 1
        continue

    # ── Concatenate all stations into one city DataFrame ─────────────────────
    # WHY outer join: different stations have different columns.
    # outer join keeps ALL columns from ALL stations — missing ones become NaN.
    df_city = pd.concat(city_station_dfs, axis=0, join='outer', ignore_index=True)

    # Sort by date so aggregation is in chronological order
    df_city.sort_values('From Date', inplace=True)
    df_city.reset_index(drop=True, inplace=True)

    # ── Aggregate hourly → monthly ────────────────────────────────────────────
    n_stations_actual = len(city_station_dfs)   # May be fewer than expected if files missing
    monthly_df = aggregate_hourly_to_monthly(df_city, city_name, n_stations_actual)

    if len(monthly_df) == 0:
        print(f"  ⚠️  SKIP {city_name}: aggregation produced 0 rows")
        cities_skipped += 1
        continue

    # ── Assign data quality tier ──────────────────────────────────────────────
    tier, actual_years, first_year, last_year = assign_tier(monthly_df)

    # ── Get state for this city ───────────────────────────────────────────────
    # All stations for one city belong to the same state — take the first one
    city_state = station_to_meta[station_ids[0]]['state']

    # ── Apply state name standardisation ─────────────────────────────────────
    # WHY: Must match disease dataset state names for Phase 3 join to work
    city_state_standardised = STATE_NAME_MAP.get(city_state, city_state)

    # ── Add metadata columns to monthly DataFrame ────────────────────────────
    monthly_df.insert(0, 'state_original',    city_state)             # Original name (for reference)
    monthly_df.insert(0, 'state',             city_state_standardised) # Standardised name
    monthly_df.insert(0, 'city',              city_name)
    monthly_df['data_quality_tier'] = tier
    monthly_df['actual_data_years'] = actual_years
    monthly_df['data_first_year']   = first_year
    monthly_df['data_last_year']    = last_year
    monthly_df['n_stations']        = n_stations_actual

    # ── Save per-city CSV ─────────────────────────────────────────────────────
    # WHY: We save individual city files so team members can work with one city
    # at a time without loading the full master CSV (which may be large)
    safe_city_name = city_name.replace('/', '_').replace(' ', '_')  # Make filename safe
    city_csv_path  = FINAL_DATASET_DIR / f"{safe_city_name}.csv"
    monthly_df.to_csv(city_csv_path, index=False)

    # ── Collect for master CSV ────────────────────────────────────────────────
    all_city_monthly_dfs.append(monthly_df)
    cities_processed += 1

    # Print progress every 25 cities (or for first/last)
    if (city_idx + 1) % 25 == 0 or city_idx == 0 or city_idx == n_total_cities - 1:
        print(f"  [{city_idx+1:3d}/{n_total_cities}] {city_name:35s} | "
              f"{tier} | {actual_years}yr | {first_year}–{last_year} | "
              f"{len(monthly_df)} months")

print()
print("=" * 60)
print(f"✅ Main loop complete")
print(f"   Cities processed    : {cities_processed}")
print(f"   Cities skipped      : {cities_skipped}")
if stations_not_found:
    print(f"   Station files missing: {stations_not_found}")


Starting main loop: 241 cities to process
  [  1/241] Agartala                            | tier_3 | 4yr | 2020–2023 | 29 months
  [ 25/241] Barmer                              | tier_4 | 1yr | 2023–2023 | 2 months
  [ 50/241] Chengalpattu                        | tier_4 | 2yr | 2022–2023 | 8 months
  [ 75/241] Ernakulam                           | tier_3 | 3yr | 2020–2022 | 39 months
  [100/241] Hyderabad                           | tier_2 | 7yr | 2017–2023 | 148 months
  [125/241] Keonjhar                            | tier_4 | 2yr | 2022–2023 | 5 months
  [150/241] Meerut                              | tier_2 | 5yr | 2019–2023 | 46 months
  [175/241] Patiala                             | tier_2 | 6yr | 2018–2023 | 62 months
  [200/241] Sasaram                             | tier_3 | 3yr | 2021–2023 | 21 months
  [225/241] Tirupati                            | tier_1 | 8yr | 2016–2023 | 81 months
  [241/241] Yamuna Nagar                        | tier_2 | 5yr | 2019–2023 | 51 months

✅ 

In [30]:
# ── Combine all city monthly DataFrames ───
print("Combining all city DataFrames into master CSV...")

master_df = pd.concat(all_city_monthly_dfs, axis=0, join='outer', ignore_index=True)

# Sort by state, city, year, month for clean ordering
master_df.sort_values(['state', 'city', 'year', 'month'], inplace=True)
master_df.reset_index(drop=True, inplace=True)

print(f"\n✅ Master DataFrame created")
print(f"   Total rows    : {len(master_df):,}  (city-month combinations)")
print(f"   Total columns : {len(master_df.columns)}")
print(f"   Cities        : {master_df['city'].nunique()}")
print(f"   States        : {master_df['state'].nunique()}")
print(f"   Year range    : {master_df['year'].min()} – {master_df['year'].max()}")

# ── Tier distribution summary ───
print("\n── Data Quality Tier Distribution ──────────────────────────")
tier_summary = (
    master_df[['city', 'data_quality_tier', 'actual_data_years',
               'data_first_year', 'data_last_year']]
    .drop_duplicates(subset=['city'])
    .groupby('data_quality_tier')
    .agg(
        cities         = ('city',             'count'),
        avg_years      = ('actual_data_years', 'mean'),
        min_first_year = ('data_first_year',   'min'),
        max_last_year  = ('data_last_year',    'max'),
    )
    .round(1)
)
print(tier_summary.to_string())

# ── State standardisation check ────
print("\n── State Name Standardisation Check ────────────────────────")
renamed_states = master_df[master_df['state'] != master_df['state_original']][
    ['state_original', 'state']
].drop_duplicates()
if len(renamed_states) > 0:
    print("States renamed (pollution → disease standard):")
    for _, r in renamed_states.iterrows():
        print(f"  '{r['state_original']}'  →  '{r['state']}'")
else:
    print("No state renames were needed (or STATE_NAME_MAP not applied).")

# ── Save master CSV ───
master_csv_path = OUTPUT_DIR / "air_pollution_master.csv"
master_df.to_csv(master_csv_path, index=False)
print(f"\n✅ Master CSV saved → {master_csv_path}")
print(f"   File size estimate: ~{len(master_df) * len(master_df.columns) * 8 / 1e6:.1f} MB")


Combining all city DataFrames into master CSV...

✅ Master DataFrame created
   Total rows    : 10,295  (city-month combinations)
   Total columns : 97
   Cities        : 241
   States        : 30
   Year range    : 2010 – 2023

── Data Quality Tier Distribution ──────────────────────────
                   cities  avg_years min_first_year max_last_year
data_quality_tier                                                
tier_1                 15        8.8           2013          2023
tier_2                 90        5.9           2016          2023
tier_3                 63        3.4           2019          2023
tier_4                 73        1.5           2019          2023

── State Name Standardisation Check ────────────────────────
States renamed (pollution → disease standard):
  'Jammu and Kashmir'  →  'Jammu & Kashmir and Ladakh'
  'Chandigarh'  →  'Other Union Territories'
  'Puducherry'  →  'Other Union Territories'

✅ Master CSV saved → ..\outputs\air_pollution_master.csv
  

Quality Check

In [32]:
# ── Main processing loop ─────
all_city_monthly_dfs = []   
# Track statistics for the summary report
cities_processed   = 0
cities_skipped     = 0
stations_not_found = []

city_list = sorted(city_to_stations.keys())   # Sort alphabetically for reproducibility
n_total_cities = len(city_list)

print(f"Starting main loop: {n_total_cities} cities to process")
print("=" * 60)

for city_idx, city_name in enumerate(city_list):

    station_ids = city_to_stations[city_name]
    n_stations  = len(station_ids)

    # ── Load all stations for this city ─────
    city_station_dfs = []

    for station_id in station_ids:
        df_station = load_and_clean_station(station_id)
        if df_station is not None:
            city_station_dfs.append(df_station)
        else:
            stations_not_found.append(station_id)

    if len(city_station_dfs) == 0:
        print(f"SKIP {city_name}: no station files found")
        cities_skipped += 1
        continue


    df_city = pd.concat(city_station_dfs, axis=0, join='outer', ignore_index=True)

    df_city.sort_values('From Date', inplace=True)
    df_city.reset_index(drop=True, inplace=True)

    # ── Aggregate hourly → monthly ────
    n_stations_actual = len(city_station_dfs)   
    monthly_df = aggregate_hourly_to_monthly(df_city, city_name, n_stations_actual)

    if len(monthly_df) == 0:
        print(f"  ⚠️  SKIP {city_name}: aggregation produced 0 rows")
        cities_skipped += 1
        continue

    # ── Assign data quality tier ────
    tier, actual_years, first_year, last_year = assign_tier(monthly_df)

    # ── Get state for this city ───

    city_state = station_to_meta[station_ids[0]]['state']


    city_state_standardised = STATE_NAME_MAP.get(city_state, city_state)

    # ── Add metadata columns to monthly DataFrame ──
    monthly_df.insert(0, 'state_original',    city_state)             # Original name (for reference)
    monthly_df.insert(0, 'state',             city_state_standardised) # Standardised name
    monthly_df.insert(0, 'city',              city_name)
    monthly_df['data_quality_tier'] = tier
    monthly_df['actual_data_years'] = actual_years
    monthly_df['data_first_year']   = first_year
    monthly_df['data_last_year']    = last_year
    monthly_df['n_stations']        = n_stations_actual

    # ── Save per-city CSV ───
    # We save individual city files so team members can work with one city
    # at a time without loading the full master CSV (which may be large)
    safe_city_name = city_name.replace('/', '_').replace(' ', '_')  # Make filename safe
    city_csv_path  = FINAL_DATASET_DIR / f"{safe_city_name}.csv"
    monthly_df.to_csv(city_csv_path, index=False)

    # ── Collect for master CSV ──
    all_city_monthly_dfs.append(monthly_df)
    cities_processed += 1

    # Print progress every 25 cities (or for first/last)
    if (city_idx + 1) % 25 == 0 or city_idx == 0 or city_idx == n_total_cities - 1:
        print(f"  [{city_idx+1:3d}/{n_total_cities}] {city_name:35s} | "
              f"{tier} | {actual_years}yr | {first_year}–{last_year} | "
              f"{len(monthly_df)} months")

print()
print("=" * 60)
print(f"✅ Main loop complete")
print(f"   Cities processed    : {cities_processed}")
print(f"   Cities skipped      : {cities_skipped}")
if stations_not_found:
    print(f"   Station files missing: {stations_not_found}")


Starting main loop: 241 cities to process
  [  1/241] Agartala                            | tier_3 | 4yr | 2020–2023 | 29 months
  [ 25/241] Barmer                              | tier_4 | 1yr | 2023–2023 | 2 months
  [ 50/241] Chengalpattu                        | tier_4 | 2yr | 2022–2023 | 8 months
  [ 75/241] Ernakulam                           | tier_3 | 3yr | 2020–2022 | 39 months
  [100/241] Hyderabad                           | tier_2 | 7yr | 2017–2023 | 148 months
  [125/241] Keonjhar                            | tier_4 | 2yr | 2022–2023 | 5 months
  [150/241] Meerut                              | tier_2 | 5yr | 2019–2023 | 46 months
  [175/241] Patiala                             | tier_2 | 6yr | 2018–2023 | 62 months
  [200/241] Sasaram                             | tier_3 | 3yr | 2021–2023 | 21 months
  [225/241] Tirupati                            | tier_1 | 8yr | 2016–2023 | 81 months
  [241/241] Yamuna Nagar                        | tier_2 | 5yr | 2019–2023 | 51 months

✅ 

Phase 1 Summary

In [33]:
print("=" * 60)
print("PHASE 1 COMPLETE — SUMMARY")
print("=" * 60)
print()
print(f"Output files saved:")
print(f"Master CSV  : outputs/air_pollution_master.csv")
print(f"Rows        : {len(master_df):,}")
print(f"Cities      : {master_df['city'].nunique()}")
print(f"Date range  : {int(master_df['year'].min())} – {int(master_df['year'].max())}")
print()
print(f"Per-city CSVs: outputs/final_dataset/")
print(f"Files saved : {cities_processed}")
print()

# Tier breakdown
print("Data quality tier breakdown:")
tier_city_counts = (
    master_df[['city', 'data_quality_tier']]
    .drop_duplicates()
    .groupby('data_quality_tier')
    .size()
)
tier_descriptions = {
    'tier_1': '≥8yr  — full ML + trend eligible',
    'tier_2': '5-7yr — trend + limited ML',
    'tier_3': '3-4yr — snapshot only',
    'tier_4': '<3yr  — flagged, excluded from ML',
}
for tier, count in sorted(tier_city_counts.items()):
    desc = tier_descriptions.get(tier, '')
    print(f"  {tier} ({desc}): {count} cities")

print()
print("Next steps:")
print("  → Phase 2 (phase_2_disease_clean.ipynb):")
print("    Load the 5 IHME GBD disease CSV files, clean, combine,")
print("    add evidence_strength labels, save outputs/disease_master.csv")
print()
print("  → Phase 4 (phase_4_aqi.ipynb) can run in parallel with Phase 2:")
print("    Compute CPCB AQI scores from the master CSV produced here.")
print()
print("IMPORTANT REMINDERS:")
print("  - air_pollution_master.csv is at CITY-MONTH level")
print("  - Disease data is at STATE-YEAR level")
print("  - Phase 3 aggregates city-month → state-year before the join")
print("  - State names in master CSV are already standardised for Phase 3")


PHASE 1 COMPLETE — SUMMARY

Output files saved:
Master CSV  : outputs/air_pollution_master.csv
Rows        : 10,295
Cities      : 241
Date range  : 2010 – 2023

Per-city CSVs: outputs/final_dataset/
Files saved : 241

Data quality tier breakdown:
  tier_1 (≥8yr  — full ML + trend eligible): 15 cities
  tier_2 (5-7yr — trend + limited ML): 90 cities
  tier_3 (3-4yr — snapshot only): 63 cities
  tier_4 (<3yr  — flagged, excluded from ML): 73 cities

Next steps:
  → Phase 2 (phase_2_disease_clean.ipynb):
    Load the 5 IHME GBD disease CSV files, clean, combine,
    add evidence_strength labels, save outputs/disease_master.csv

  → Phase 4 (phase_4_aqi.ipynb) can run in parallel with Phase 2:
    Compute CPCB AQI scores from the master CSV produced here.

IMPORTANT REMINDERS:
  - air_pollution_master.csv is at CITY-MONTH level
  - Disease data is at STATE-YEAR level
  - Phase 3 aggregates city-month → state-year before the join
  - State names in master CSV are already standardised for Ph